# 영화 포스터 웹크롤링 저장하기
1. 해당 사이트에서 개발자도구(F12) 활용해 페이지 구조 확인
   -. copy element -> HTML 소스코드 구조 확인
      <img src="/thumb?width=178&amp;height=267&amp;m_code=20242837&amp;source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg" alt="왕과 사는 남자">
      HTML 구조: <img src="이미지URL" alt="영화제목">
   -. src가 thumb?...로 시작 >> 상대경로 >> 실제 요청 URL은 도메인을 붙여서 만들어야 함 >> https://www.moviechart.co.kr/thumb?width=...
   -. HTML에서 &amp: -> 보통 파싱하면 자동으로 &로 복원되나 확인 필요
2. 순위별 포스터 img 목록 한 번에 뽑기 >> URL이 제대로 뽑히는지 출력해서 검증하는 단계
   -. 페이지를 request로 가져오기
   -. img 태그 중에서 포스터에 해당하는 것만 골라오기
   -. (title=alt, image_url=src) 형태로 리스트 생성하기
   >> alt="영화제목" 조건으로 목록을 뽑으니 영화 외 다른 정보까지 추출됨
   >> src에 /thumb?width=178&height=267이 포함된 조건으로 코드 수정
   >> starswith("/thumb") 사용하여 포스터 전용 URL 패턴과 정확히 일치하는 것만 추출함
3. 포스터 URL로 실제 이미지 다운로드 -> 파일명 정리 -> 폴더 저장
   -. 현재 URL: 리사이즈된 썸네일 요청 URL -> sorce= 뒤에 있는 원본 이미지 URL을 직접 사용하는 것이 좋음(a. 화질이 더 좋음. b. 서버 부담 감소. c. 구조 명확함)
   -. 단계: 1) source 파라미터에서 원본 url 추출
           2) 파일명 안전하게 정리 : / \ : * ? " < > |, 공백 -> 파일명에 사용할 수 없는 문자 제거
           3) 저장

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.moviechart.co.kr"
URL  = "https://www.moviechart.co.kr/rank/realtime/index/image"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(URL, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

poster_imgs = []

for img in soup.select("img"):
    src = img.get("src", "")
    
    if src.startswith("/thumb"):
        poster_imgs.append(img)

print("포스터 개수:", len(poster_imgs))

for i, img in enumerate(poster_imgs[:10], start=1):
    title = img.get("alt")
    src = img.get("src")
    full_url = urljoin(BASE, src)

    print(f"{i}. {title}")
    print("   ", full_url)

포스터 개수: 20
1. 왕과 사는 남자
    https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20242837&source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg
2. 휴민트
    https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20241266&source=https://admin.moviechart.co.kr/assets/upload/movie/260112024651_6301.jpg
3. 초속 5센티미터
    https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20259583&source=https://admin.moviechart.co.kr/assets/upload/movie/260213052538_7190.jpg
4. 너자 2
    https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20261181&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063756_4036.jpg
5. 슬라이드 스트럼 뮤트
    https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20261450&source=https://admin.moviechart.co.kr/assets/upload/movie/260219042530_2928.jpg
6. 넘버원
    https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20252373&source=https://admin.moviechart.co.kr/assets/upload/movie/260123065552_5

In [5]:
import os
import re
import requests
from urllib.parse import urlparse, parse_qs

SAVE_DIR = "movie_posters"
os.makedirs(SAVE_DIR, exist_ok=True)

for i, img in enumerate(poster_imgs, start=1):
    
    title = img.get("alt", "").strip()
    thumb_url = urljoin(BASE, img.get("src"))
    
    # 1️⃣ source 파라미터 추출
    parsed = urlparse(thumb_url)
    query_params = parse_qs(parsed.query)
    original_url = query_params.get("source", [None])[0]
    
    if not original_url:
        continue
    
    # 2️⃣ 파일명 정리
    safe_title = re.sub(r'[\\/:*?"<>|]', "", title)
    safe_title = safe_title.replace(" ", "_")
    
    filename = f"{i:02d}_{safe_title}.jpg"
    filepath = os.path.join(SAVE_DIR, filename)
    
    # 3️⃣ 이미지 다운로드
    img_response = requests.get(original_url, headers=headers)
    
    with open(filepath, "wb") as f:
        f.write(img_response.content)
    
    print(f"{filename} 저장 완료")

01_왕과_사는_남자.jpg 저장 완료
02_휴민트.jpg 저장 완료
03_초속_5센티미터.jpg 저장 완료
04_너자_2.jpg 저장 완료
05_슬라이드_스트럼_뮤트.jpg 저장 완료
06_넘버원.jpg 저장 완료
07_햄넷.jpg 저장 완료
08_부흥.jpg 저장 완료
09_매드_댄스_오피스.jpg 저장 완료
10_신의악단.jpg 저장 완료
11_렌탈_패밀리_가족을_빌려드립니다.jpg 저장 완료
12_몬테크리스토_백작.jpg 저장 완료
13_호퍼스.jpg 저장 완료
14_만약에_우리.jpg 저장 완료
15_프랑켄슈타인.jpg 저장 완료
16_폭풍의_언덕.jpg 저장 완료
17_점보.jpg 저장 완료
18_아기_티라노_디보_초식이지만_괜찮아!.jpg 저장 완료
19_직사각형,_삼각형.jpg 저장 완료
20_아바타_불과_재.jpg 저장 완료
